# Phân khúc khách hàng Online Retail II

Pipeline: làm sạch → EDA → RFM features → baseline ngũ phân vị → K-Means & Hierarchical → PCA.

**Câu hỏi nghiên cứu:** Từ RFM và hành vi mua, khách tách thành nhóm nào và nhóm nào đáng giữ / kích hoạt lại?


In [ ]:
from pathlib import Path
import os
import warnings

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

XLSX = Path(os.environ.get(
    "RETAIL_XLSX",
    r"C:/Users/nguye/Downloads/online+retail+ii/online_retail_II.xlsx",
))
DB = Path(os.environ.get(
    "RETAIL_DB",
    r"C:/Users/nguye/Downloads/online-retail-ii/retail.duckdb",
))
SNAPSHOT = "2011-12-10"
print("XLSX exists:", XLSX.exists())
print("DB exists:", DB.exists())

## 1. Nạp và làm sạch dữ liệu


In [ ]:
con = duckdb.connect(str(DB))
tables = set(con.execute("SHOW TABLES").df()["name"].tolist())

if {"purchases", "cancellations"}.issubset(tables):
    print("Dùng bảng purchases/cancellations có sẵn trong DuckDB")
else:
    if not XLSX.exists():
        raise FileNotFoundError(f"Không thấy Excel: {XLSX}")
    print("Đang đọc Excel (có thể mất 1–2 phút)...")
    con.execute("INSTALL excel; LOAD excel;")
    x = XLSX.as_posix()
    con.execute(f'''
        CREATE OR REPLACE TABLE retail AS
        SELECT
            Invoice AS invoice,
            StockCode AS stock_code,
            NULLIF(Description, '') AS description,
            TRY_CAST(Quantity AS INTEGER) AS quantity,
            CAST('1899-12-30' AS TIMESTAMP)
                + TRY_CAST(InvoiceDate AS DOUBLE) * INTERVAL 1 DAY AS invoice_date,
            TRY_CAST(Price AS DOUBLE) AS price,
            TRY_CAST(NULLIF(TRIM("Customer ID"), '') AS BIGINT) AS customer_id,
            Country AS country
        FROM (
            SELECT * FROM read_xlsx('{x}', sheet = 'Year 2009-2010', all_varchar = true)
            UNION ALL
            SELECT * FROM read_xlsx('{x}', sheet = 'Year 2010-2011', all_varchar = true)
        )
    ''')
    con.execute('''
        CREATE OR REPLACE TABLE purchases AS
        SELECT DISTINCT
            invoice, stock_code, description, quantity, invoice_date, price,
            customer_id, country, quantity * price AS line_revenue
        FROM retail
        WHERE customer_id IS NOT NULL
          AND quantity > 0 AND price > 0
          AND invoice NOT LIKE 'C%'
          AND UPPER(stock_code) NOT IN ('POST', 'DOT', 'M', 'D', 'S', 'B')
    ''')
    con.execute('''
        CREATE OR REPLACE TABLE cancellations AS
        SELECT DISTINCT invoice, stock_code, quantity, invoice_date, price, customer_id
        FROM retail
        WHERE customer_id IS NOT NULL
          AND (invoice LIKE 'C%' OR quantity < 0)
    ''')

print(con.execute('''
    SELECT
        (SELECT COUNT(*) FROM purchases) AS purchase_rows,
        (SELECT COUNT(*) FROM cancellations) AS cancel_rows,
        (SELECT COUNT(DISTINCT customer_id) FROM purchases) AS customers
''').df())

## 2. EDA ngắn


In [ ]:
print(con.execute('''
    SELECT
        COUNT(DISTINCT invoice) AS invoices,
        COUNT(DISTINCT stock_code) AS products,
        MIN(invoice_date) AS first_date,
        MAX(invoice_date) AS last_date,
        ROUND(SUM(line_revenue), 2) AS revenue
    FROM purchases
''').df())

monthly = con.execute('''
    SELECT STRFTIME(invoice_date, '%Y-%m') AS month,
           ROUND(SUM(line_revenue), 2) AS revenue
    FROM purchases
    GROUP BY 1 ORDER BY 1
''').df()
ax = monthly.plot(x='month', y='revenue', figsize=(10, 3), legend=False, title='Doanh thu theo tháng')
ax.set_ylabel('GBP')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'eda_revenue_monthly.png', dpi=120)
plt.show()

## 3. Đặc trưng khách hàng (RFM + hành vi)


In [ ]:
con.execute(f'''
    CREATE OR REPLACE TABLE customer_features AS
    WITH purchase_agg AS (
        SELECT
            customer_id,
            DATE_DIFF('day', MAX(invoice_date), TIMESTAMP '{SNAPSHOT}') AS recency,
            COUNT(DISTINCT invoice) AS frequency,
            ROUND(SUM(line_revenue), 2) AS monetary,
            ROUND(SUM(line_revenue) / COUNT(DISTINCT invoice), 2) AS avg_order,
            COUNT(DISTINCT stock_code) AS n_products,
            DATE_DIFF('day', MIN(invoice_date), MAX(invoice_date)) AS tenure_days
        FROM purchases
        GROUP BY customer_id
    ),
    cancel_agg AS (
        SELECT customer_id,
               ROUND(SUM(ABS(quantity) * price), 2) AS cancel_value,
               COUNT(DISTINCT invoice) AS cancel_invoices
        FROM cancellations
        GROUP BY customer_id
    )
    SELECT
        p.*,
        COALESCE(c.cancel_value, 0) AS cancel_value,
        COALESCE(c.cancel_invoices, 0) AS cancel_invoices,
        ROUND(
            COALESCE(c.cancel_value, 0)
            / NULLIF(p.monetary + COALESCE(c.cancel_value, 0), 0), 4
        ) AS cancel_rate
    FROM purchase_agg p
    LEFT JOIN cancel_agg c USING (customer_id)
''')

features = con.execute('SELECT * FROM customer_features ORDER BY customer_id').df()
print(features.describe().round(2))
features.to_excel(OUT / 'customer_features.xlsx', index=False)
features.head()

## 4. Baseline RFM ngũ phân vị


In [ ]:
rfm = features[['customer_id', 'recency', 'frequency', 'monetary']].copy()
rfm['R'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_score'] = rfm['R'] + rfm['F'] + rfm['M']

def name_rfm(row):
    if row['R'] >= 4 and row['F'] >= 4 and row['M'] >= 4:
        return 'VIP_trung_thanh'
    if row['R'] >= 4 and row['F'] <= 2:
        return 'khach_moi_tiem_nang'
    if row['R'] >= 3 and row['F'] >= 3 and row['M'] >= 3:
        return 'khach_trung_thanh'
    if row['R'] <= 2 and row['F'] >= 4 and row['M'] >= 4:
        return 'sap_roi_gia_tri_cao'
    if row['R'] <= 2 and row['F'] <= 2:
        return 'nguy_co_mat'
    if row['R'] >= 4:
        return 'moi_mua_gan_day'
    if row['R'] <= 2:
        return 'lau_khong_mua'
    return 'trung_binh'

rfm['rfm_segment'] = rfm.apply(name_rfm, axis=1)
rfm_summary = (
    rfm.groupby('rfm_segment')
      .agg(so_khach=('customer_id', 'count'),
           median_recency=('recency', 'median'),
           median_frequency=('frequency', 'median'),
           median_monetary=('monetary', 'median'),
           tong_doanh_thu=('monetary', 'sum'))
      .sort_values('tong_doanh_thu', ascending=False)
)
rfm.to_excel(OUT / 'rfm_baseline.xlsx', index=False)
rfm_summary.to_excel(OUT / 'rfm_baseline_summary.xlsx')
rfm_summary

## 5. Chuẩn hóa + K-Means & Hierarchical


In [ ]:
log_cols = ['frequency', 'monetary', 'avg_order', 'n_products', 'cancel_value', 'cancel_invoices']
X = features[['recency', 'tenure_days', 'cancel_rate'] + log_cols].copy()
for c in log_cols:
    X[c] = np.log1p(X[c])
X_scaled = StandardScaler().fit_transform(X)

rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_scaled)
    hi = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_scaled)
    rows.append({
        'k': k,
        'kmeans_silhouette': silhouette_score(X_scaled, km),
        'kmeans_db': davies_bouldin_score(X_scaled, km),
        'hier_silhouette': silhouette_score(X_scaled, hi),
        'hier_db': davies_bouldin_score(X_scaled, hi),
    })
compare = pd.DataFrame(rows).round(4)
compare.to_excel(OUT / 'cluster_algorithm_comparison.xlsx', index=False)
compare

## 6. K-Means k=4 — hồ sơ cụm + PCA


In [ ]:
NAMES = {
    0: 'khach_mot_lan_roi_bo',
    1: 'khach_thuong_xuyen',
    2: 'khach_vip_trung_thanh',
    3: 'khach_rui_ro_hay_huy',
}

labels = KMeans(n_clusters=4, random_state=42, n_init=20).fit_predict(X_scaled)
result = features.copy()
result['cluster'] = labels
result['ten_cum'] = result['cluster'].map(NAMES)

profile = (
    result.groupby(['cluster', 'ten_cum'])
          .agg(so_khach=('customer_id', 'count'),
               median_recency=('recency', 'median'),
               median_frequency=('frequency', 'median'),
               median_monetary=('monetary', 'median'),
               median_cancel_rate=('cancel_rate', 'median'),
               tong_doanh_thu=('monetary', 'sum'))
          .reset_index()
          .sort_values('tong_doanh_thu', ascending=False)
)
print(profile.to_string(index=False))

pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(X_scaled)
print('Explained variance:', pca.explained_variance_ratio_.round(3), 'sum=', round(pca.explained_variance_ratio_.sum(), 3))

fig, ax = plt.subplots(figsize=(8, 5))
for cid, name in NAMES.items():
    m = result['cluster'] == cid
    ax.scatter(xy[m, 0], xy[m, 1], s=10, alpha=0.55, label=name)
ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
ax.set_title('K-Means k=4 trên PCA')
ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig(FIG / 'pca_clusters_k4.png', dpi=150)
plt.show()

result.to_excel(OUT / 'customer_clusters_k4.xlsx', index=False)
profile.to_excel(OUT / 'cluster_summary_k4.xlsx', index=False)

## 7. Kết luận ngắn

- Baseline RFM đã chỉ ra VIP và nhóm nguy cơ mất.
- **K-Means k=4** tốt hơn Hierarchical ở cùng k (Silhouette / Davies–Bouldin).
- Hành động: giữ VIP, upsell nhóm thường xuyên, win-back nhóm một lần, giảm ma sát nhóm hay hủy.

Chi tiết xem `bao_cao.md` và `slides.html`.
